# Flash Evaluation on DARPA E3 Cadets Dataset: 

This notebook is specifically designed for the evaluation of Flash on the DARPA E3 Cadets dataset. Notably, the Cadets dataset is characterized as a node-level dataset. In our analysis, Flash is configured to operate in a node-level setting to aptly assess this dataset. A key aspect to note is that the Cadets dataset lacks certain essential node attributes for specific node types. This limitation means that Flash cannot be operated in a decoupled mode with offline GNN embeddings for this dataset. Consequently, we employ an online GNN coupled with word2vec semantic embeddings to achieve effective evaluation results for this dataset.

## Dataset Access: 
- Access the Cadets dataset via the following link: [Cadets Dataset](https://drive.google.com/drive/folders/1QlbUFWAGq3Hpl8wVdzOdIoZLFxkII4EK).
- The dataset files will be downloaded automatically by the script.

## Data Parsing and Execution:
- The script is designed to automatically parse the downloaded data files.
- Execute all cells within this notebook to obtain the evaluation results.

## Model Training and Execution Flexibility:
- The notebook is configured to use pre-trained model weights by default.
- It also provides the option to set parameters for independently training Graph Neural Networks (GNNs) and word2vec models.
- These newly trained models can then be utilized for a comprehensive evaluation of the dataset.

Adhere to these steps for a detailed and effective analysis of the Cadets dataset using Flash.


In [1]:
from rich.console import Console
from rich.theme import Theme
from rich.logging import RichHandler
from rich.progress import Progress, SpinnerColumn, BarColumn, TextColumn, TimeRemainingColumn
import logging
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

custom_theme = Theme({
    "info": "cyan",
    "warning": "yellow",
    "error": "bold red",
    "debug": "dim",
    "success": "bold green",
    "step": "bold magenta"
})

console = Console(theme=custom_theme)

logging.basicConfig(
    level=logging.INFO,
    format="%(message)s",
    datefmt="[%X]",
    handlers=[RichHandler(console=console, rich_tracebacks=True)]
)

logger = logging.getLogger("flash-cadets")
logger.setLevel(logging.INFO)

console.print("[bold cyan]Flash-IDS: Cadets Dataset Evaluation[/bold cyan]", style="success")
logger.info(f"Device: {device}")
logger.info("Initializing...")

Flash-IDS: Cadets Dataset Evaluation

[16:42:03] INFO     Device: cuda                                                                   ]8;id=369597;file:///tmp/ipykernel_23639/1839788288.py\1839788288.py]8;;\:]8;id=154411;file:///tmp/ipykernel_23639/1839788288.py#32\32]8;;\

           INFO     Initializing...                                                                ]8;id=959190;file:///tmp/ipykernel_23639/1839788288.py\1839788288.py]8;;\:]8;id=444627;file:///tmp/ipykernel_23639/1839788288.py#33\33]8;;\

In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from torch_geometric.data import Data
import os
import torch.nn.functional as F
import orjson as json
import warnings
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
warnings.filterwarnings('ignore')
from torch_geometric.loader import NeighborLoader
import multiprocessing

%matplotlib inline

In [3]:
Train = False

In [4]:
from pprint import pprint
import gzip
from sklearn.manifold import TSNE
import json
import copy
import os

In [5]:
import re

def extract_uuid(line):
    pattern_uuid = re.compile(r'uuid\":\"(.*?)\"')
    return pattern_uuid.findall(line)

def extract_subject_type(line):
    pattern_type = re.compile(r'type\":\"(.*?)\"')
    return pattern_type.findall(line)

def show(file_path):
    logger.info(f"Processing: {file_path}")

def extract_edge_info(line):
    pattern_src = re.compile(r'subject\":{\"com.bbn.tc.schema.avro.cdm18.UUID\":\"(.*?)\"}')
    pattern_dst1 = re.compile(r'predicateObject\":{\"com.bbn.tc.schema.avro.cdm18.UUID\":\"(.*?)\"}')
    pattern_dst2 = re.compile(r'predicateObject2\":{\"com.bbn.tc.schema.avro.cdm18.UUID\":\"(.*?)\"}')
    pattern_type = re.compile(r'type\":\"(.*?)\"')
    pattern_time = re.compile(r'timestampNanos\":(.*?),')

    edge_type = extract_subject_type(line)[0]
    timestamp = pattern_time.findall(line)[0]
    src_id = pattern_src.findall(line)

    if len(src_id) == 0:
        return None, None, None, None, None

    src_id = src_id[0]
    dst_id1 = pattern_dst1.findall(line)
    dst_id2 = pattern_dst2.findall(line)

    if len(dst_id1) > 0 and dst_id1[0] != 'null':
        dst_id1 = dst_id1[0]
    else:
        dst_id1 = None

    if len(dst_id2) > 0 and dst_id2[0] != 'null':
        dst_id2 = dst_id2[0]
    else:
        dst_id2 = None

    return src_id, edge_type, timestamp, dst_id1, dst_id2

def process_data(file_path):
    id_nodetype_map = {}
    notice_num = 1000000
    logger.info(f"Starting data processing: {file_path}")
    for i in range(100):
        now_path = file_path + '.' + str(i)
        if i == 0:
            now_path = file_path
        if not os.path.exists(now_path):
            break

        with open(now_path, 'r') as f:
            show(now_path)
            cnt = 0
            total_lines = 0
            for line in f:
                cnt += 1
                total_lines += 1
                if cnt % notice_num == 0:
                    logger.debug(f"Processed {cnt:,} lines...")

                if 'com.bbn.tc.schema.avro.cdm18.Event' in line or 'com.bbn.tc.schema.avro.cdm18.Host' in line:
                    continue

                if 'com.bbn.tc.schema.avro.cdm18.TimeMarker' in line or 'com.bbn.tc.schema.avro.cdm18.StartMarker' in line:
                    continue

                if 'com.bbn.tc.schema.avro.cdm18.UnitDependency' in line or 'com.bbn.tc.schema.avro.cdm18.EndMarker' in line:
                    continue

                uuid = extract_uuid(line)[0]
                subject_type = extract_subject_type(line)

                if len(subject_type) < 1:
                    if 'com.bbn.tc.schema.avro.cdm18.MemoryObject' in line:
                        id_nodetype_map[uuid] = 'MemoryObject'
                        continue
                    if 'com.bbn.tc.schema.avro.cdm18.NetFlowObject' in line:
                        id_nodetype_map[uuid] = 'NetFlowObject'
                        continue
                    if 'com.bbn.tc.schema.avro.cdm18.UnnamedPipeObject' in line:
                        id_nodetype_map[uuid] = 'UnnamedPipeObject'
                        continue

                id_nodetype_map[uuid] = subject_type[0]

    logger.info(f"Data processing complete: {len(id_nodetype_map)} nodes extracted")
    return id_nodetype_map

def process_edges(file_path, id_nodetype_map):
    notice_num = 1000000
    not_in_cnt = 0
    logger.info(f"Starting edge processing: {file_path}")

    for i in range(100):
        now_path = file_path + '.' + str(i)
        if i == 0:
            now_path = file_path
        if not os.path.exists(now_path):
            break

        with open(now_path, 'r') as f, open(now_path+'.txt', 'w') as fw:
            cnt = 0
            for line in f:
                cnt += 1
                if cnt % notice_num == 0:
                    logger.debug(f"Processed {cnt:,} edges...")

                if 'com.bbn.tc.schema.avro.cdm18.Event' in line:
                    src_id, edge_type, timestamp, dst_id1, dst_id2 = extract_edge_info(line)

                    if src_id is None or src_id not in id_nodetype_map:
                        not_in_cnt += 1
                        continue

                    src_type = id_nodetype_map[src_id]

                    if dst_id1 is not None and dst_id1 in id_nodetype_map:
                        dst_type1 = id_nodetype_map[dst_id1]
                        this_edge1 = f"{src_id}\t{src_type}\t{dst_id1}\t{dst_type1}\t{edge_type}\t{timestamp}\n"
                        fw.write(this_edge1)

                    if dst_id2 is not None and dst_id2 in id_nodetype_map:
                        dst_type2 = id_nodetype_map[dst_id2]
                        this_edge2 = f"{src_id}\t{src_type}\t{dst_id2}\t{dst_type2}\t{edge_type}\t{timestamp}\n"
                        fw.write(this_edge2)

# Option: Use synthetic data for smoke testing (no download required)
USE_SYNTHETIC = True  # Set to False to use real dataset (requires download)

def run_data_processing():
    if USE_SYNTHETIC:
        create_synthetic_data()
        return
    
    logger.info("Checking for dataset files...")
    
    tar_file_1 = 'ta1-cadets-e3-official.json.tar.gz'
    tar_file_2 = 'ta1-cadets-e3-official-2.json.tar.gz'
    json_file_1 = 'ta1-cadets-e3-official.json'
    json_file_2 = 'ta1-cadets-e3-official-2.json'
    
    # Check if already processed
    if os.path.exists('cadets_train.txt') and os.path.exists('cadets_test.txt'):
        logger.info("Processed data already exists! Skipping data processing.")
        return
    
    # Check for raw JSON files (already extracted)
    if os.path.exists(json_file_1) and os.path.exists(json_file_2):
        logger.info("Found extracted JSON files, processing...")
    elif not os.path.exists(tar_file_1) or not os.path.exists(tar_file_2):
        logger.error(f"Missing dataset files! Please download them first.")
        logger.info(f"Expected files: {tar_file_1}, {tar_file_2}")
        logger.info("Or download from: https://drive.google.com/drive/folders/1QlbUFWAGq3Hpl8wVdzOdIoZLFxkII4EK")
        return
    else:
        logger.info("Extracting dataset archives...")
        os.system(f'tar -zxvf {tar_file_1}')
        os.system(f'tar -zxvf {tar_file_2}')

    path_list = [json_file_1, json_file_2]

    for path in path_list:
        if os.path.exists(path):
            id_nodetype_map = process_data(path)
            process_edges(path, id_nodetype_map)
        else:
            logger.warning(f"File not found: {path}")

    if os.path.exists('ta1-cadets-e3-official.json.1.txt'):
        os.system('cp ta1-cadets-e3-official.json.1.txt cadets_train.txt')
        logger.info("Created cadets_train.txt")
    else:
        logger.warning("Could not create cadets_train.txt")
    
    if os.path.exists('ta1-cadets-e3-official-2.json.txt'):
        os.system('cp ta1-cadets-e3-official-2.json.txt cadets_test.txt')
        logger.info("Created cadets_test.txt")
    else:
        logger.warning("Could not create cadets_test.txt")

def create_synthetic_data():
    """Create synthetic test data for smoke testing without downloading datasets."""
    import random
    import uuid
    
    logger.info("Creating synthetic test data (smoke test mode)...")
    
    # Generate synthetic train/test data
    # Format: actorID, actor_type, objectID, object_type, action, timestamp
    node_types = ['SUBJECT_PROCESS', 'FILE_OBJECT_FILE', 'FILE_OBJECT_UNIX_SOCKET', 
               'UnnamedPipeObject', 'NetFlowObject', 'FILE_OBJECT_DIR']
    actions = ['exec', 'open', 'read', 'write', 'fork', 'network']
    
    def generate_synthetic_data(num_lines, is_test=False):
        lines = []
        for i in range(num_lines):
            actor = f"proc_{random.randint(1, 100)}"
            obj = f"file_{random.randint(1, 100)}" if random.random() > 0.5 else f"sock_{random.randint(1, 50)}"
            actor_type = node_types[random.randint(0, 2)]
            obj_type = node_types[random.randint(3, 5)]
            action = actions[random.randint(0, len(actions)-1)]
            timestamp = str(1000000 + i)
            lines.append(f"{actor}\t{actor_type}\t{obj}\t{obj_type}\t{action}\t{timestamp}\n")
        return lines
    
    # Create train data
    train_lines = generate_synthetic_data(1000)
    with open('cadets_train.txt', 'w') as f:
        f.writelines(train_lines)
    logger.info(f"Created cadets_train.txt with {len(train_lines)} lines")
    
    # Create test data
    test_lines = generate_synthetic_data(500, is_test=True)
    with open('cadets_test.txt', 'w') as f:
        f.writelines(test_lines)
    logger.info(f"Created cadets_test.txt with {len(test_lines)} lines")
    
    # Also create ground truth for testing
    import json
    malicious = [f"proc_{i}" for i in range(1, 11)]  # First 10 processes as malicious
    with open('data_files/cadets.json', 'w') as f:
        json.dump(malicious, f)
    logger.info("Created synthetic ground truth")
    logger.info("Smoke test data ready!")

In [6]:
run_data_processing()

[16:42:05] INFO     Creating synthetic test data (smoke test mode)...                             ]8;id=273837;file:///tmp/ipykernel_23639/1797071445.py\1797071445.py]8;;\:]8;id=30423;file:///tmp/ipykernel_23639/1797071445.py#190\190]8;;\

           INFO     Created cadets_train.txt with 1000 lines                                      ]8;id=677842;file:///tmp/ipykernel_23639/1797071445.py\1797071445.py]8;;\:]8;id=455076;file:///tmp/ipykernel_23639/1797071445.py#214\214]8;;\

           INFO     Created cadets_test.txt with 500 lines                                        ]8;id=260742;file:///tmp/ipykernel_23639/1797071445.py\1797071445.py]8;;\:]8;id=425215;file:///tmp/ipykernel_23639/1797071445.py#220\220]8;;\

           INFO     Created synthetic ground truth                                                ]8;id=908220;file:///tmp/ipykernel_23639/1797071445.py\1797071445.py]8;;\:]8;id=837689;file:///tmp/ipykernel_23639/1797071445.py#227\227]8;;\

           INFO     Smoke test data ready!                                                        ]8;id=602800;file:///tmp/ipykernel_23639/1797071445.py\1797071445.py]8;;\:]8;id=791980;file:///tmp/ipykernel_23639/1797071445.py#228\228]8;;\

In [7]:
def add_node_properties(nodes, node_id, properties):
    if node_id not in nodes:
        nodes[node_id] = []
    nodes[node_id].extend(properties)

def update_edge_index(edges, edge_index, index):
    for src_id, dst_id in edges:
        src = index[src_id]
        dst = index[dst_id]
        edge_index[0].append(src)
        edge_index[1].append(dst)

def prepare_graph(df):
    nodes, labels, edges = {}, {}, []
    dummies = {'SUBJECT_PROCESS': 0, 'FILE_OBJECT_FILE': 1, 'FILE_OBJECT_UNIX_SOCKET': 2, 
               'UnnamedPipeObject': 3, 'NetFlowObject': 4, 'FILE_OBJECT_DIR': 5}
    
    for _, row in df.iterrows():
        action = row["action"]
        properties = [row['exec'], action] + ([row['path']] if row['path'] else [])
        
        actor_id = row["actorID"]
        add_node_properties(nodes, actor_id, properties)
        labels[actor_id] = dummies[row['actor_type']]

        object_id = row["objectID"]
        add_node_properties(nodes, object_id, properties)
        labels[object_id] = dummies[row['object']]

        edges.append((actor_id, object_id))

    features, feat_labels, edge_index, index_map = [], [], [[], []], {}
    for node_id, props in nodes.items():
        features.append(props)
        feat_labels.append(labels[node_id])
        index_map[node_id] = len(features) - 1

    update_edge_index(edges, edge_index, index_map)

    return features, feat_labels, edge_index, list(index_map.keys())

In [8]:
from torch_geometric.nn import GCNConv
from torch_geometric.nn import SAGEConv, GATConv
import torch.nn.functional as F
import torch.nn as nn

class GCN(torch.nn.Module):
    def __init__(self,in_channel,out_channel):
        super().__init__()
        self.conv1 = SAGEConv(in_channel, 32, normalize=True)
        self.conv2 = SAGEConv(32, out_channel, normalize=True)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = x.relu()
        x = F.dropout(x, p=0.5, training=self.training)

        x = self.conv2(x, edge_index)
        return F.softmax(x, dim=1)

In [9]:
def visualize(h, color):
    z = TSNE(n_components=2).fit_transform(h.detach().cpu().numpy())

    plt.figure(figsize=(10,10))
    plt.xticks([])
    plt.yticks([])

    plt.scatter(z[:, 0], z[:, 1], s=70, c=color, cmap="Set2")
    plt.show()

In [10]:
from gensim.models.callbacks import CallbackAny2Vec
import gensim
from gensim.models import Word2Vec
from multiprocessing import Pool
from itertools import compress
from tqdm import tqdm
import time

class EpochSaver(CallbackAny2Vec):
    '''Callback to save model after each epoch.'''

    def __init__(self):
        self.epoch = 0

    def on_epoch_end(self, model):
        model.save('word2vec_cadets_E3.model')
        self.epoch += 1

In [11]:
class EpochLogger(CallbackAny2Vec):
    '''Callback to log information about training'''

    def __init__(self):
        self.epoch = 0

    def on_epoch_begin(self, model):
        logger.debug(f"Epoch #{self.epoch} start")

    def on_epoch_end(self, model):
        logger.debug(f"Epoch #{self.epoch} end")
        self.epoch += 1

In [12]:
logger = EpochLogger()
saver = EpochSaver()

In [13]:
def add_attributes(d,p):
    """Add additional attributes. For synthetic mode, returns data as-is."""
    if USE_SYNTHETIC:
        logger.debug("Synthetic mode: skipping attribute enrichment")
        return d
    
    f = open(p)
    data = [json.loads(x) for x in f if "EVENT" in x]

    info = []
    for x in data:
        try:
            action = x['datum']['com.bbn.tc.schema.avro.cdm18.Event']['type']
        except:
            action = ''
        try:
            actor = x['datum']['com.bbn.tc.schema.avro.cdm18.Event']['subject']['com.bbn.tc.schema.avro.cdm18.UUID']
        except:
            actor = ''
        try:
            obj = x['datum']['com.bbn.tc.schema.avro.cdm18.Event']['predicateObject']['com.bbn.tc.schema.avro.cdm18.UUID']
        except:
            obj = ''
        try:
            timestamp = x['datum']['com.bbn.tc.schema.avro.cdm18.Event']['timestampNanos']
        except:
            timestamp = ''
        try:
            cmd = x['datum']['com.bbn.tc.schema.avro.cdm18.Event']['properties']['map']['exec']
        except:
            cmd = ''
        try:
            path = x['datum']['com.bbn.tc.schema.avro.cdm18.Event']['predicateObjectPath']['string']
        except:
            path = ''
        try:
            path2 = x['datum']['com.bbn.tc.schema.avro.cdm18.Event']['predicateObject2Path']['string']
        except:
            path2 = ''
        try:
            obj2 = x['datum']['com.bbn.tc.schema.avro.cdm18.Event']['predicateObject2']['com.bbn.tc.schema.avro.cdm18.UUID']
            info.append({'actorID':actor,'objectID':obj2,'action':action,'timestamp':timestamp,'exec':cmd, 'path':path2})
        except:
            pass

        info.append({'actorID':actor,'objectID':obj,'action':action,'timestamp':timestamp,'exec':cmd, 'path':path})

    rdf = pd.DataFrame.from_records(info).astype(str)
    d = d.astype(str)

    return d.merge(rdf,how='inner',on=['actorID','objectID','action','timestamp']).drop_duplicates()

In [15]:
if Train:
    f = open("cadets_train.txt")
    data = f.read().split('\n')
    data = [line.split('\t') for line in data]
    df = pd.DataFrame (data, columns = ['actorID', 'actor_type','objectID','object','action','timestamp'])
    df = df.dropna()
    df.sort_values(by='timestamp', ascending=True,inplace=True)
    df = add_attributes(df, "dummy")
    phrases,labels,edges,mapp = prepare_graph(df)
else:
    logger.info("Loading pre-trained model mode (Train=False)")
    phrases, labels, edges, mapp = None, None, None, None

AttributeError: 'EpochLogger' object has no attribute 'info'

In [16]:
from sklearn.utils import class_weight
import torch.nn.functional as F
from torch.nn import CrossEntropyLoss

model = GCN(30,6).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

In [17]:
if Train:
    word2vec = Word2Vec(sentences=phrases, vector_size=30, window=5, min_count=1, workers=8,epochs=300,callbacks=[saver,logger])

In [19]:
import math
import torch
import numpy as np
from gensim.models import Word2Vec

class PositionalEncoder:

    def __init__(self, d_model, max_len=100000):
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        self.pe = torch.zeros(max_len, d_model)
        self.pe[:, 0::2] = torch.sin(position * div_term)
        self.pe[:, 1::2] = torch.cos(position * div_term)

    def embed(self, x):
        return x + self.pe[:x.size(0)]


def infer(document):
    """Infer embeddings for a document. For synthetic mode, returns random embeddings."""
    if USE_SYNTHETIC or w2vmodel is None:
        logger.debug("Synthetic mode: using random embeddings")
        return np.random.randn(30).astype(np.float32)
    
    word_embeddings = [w2vmodel.wv[word] for word in document if word in  w2vmodel.wv]
    
    if not word_embeddings:
        return np.zeros(20)

    output_embedding = torch.tensor(word_embeddings, dtype=torch.float)
    if len(document) < 100000:
        output_embedding = encoder.embed(output_embedding)

    output_embedding = output_embedding.detach().cpu().numpy()
    return np.mean(output_embedding, axis=0)

encoder = PositionalEncoder(30)

# Load Word2Vec model (or None for synthetic mode)
if USE_SYNTHETIC:
    w2vmodel = None
    logger.info("Synthetic mode: using random embeddings instead of Word2Vec")
else:
    w2vmodel = Word2Vec.load("trained_weights/cadets/word2vec_cadets_E3.model")

AttributeError: 'EpochLogger' object has no attribute 'info'

In [20]:
from torch_geometric import utils

if Train:
    l = np.array(labels)
    class_weights = class_weight.compute_class_weight(class_weight = None,classes = np.unique(l),y = l)
    class_weights = torch.tensor(class_weights,dtype=torch.float).to(device)
    criterion = CrossEntropyLoss(weight=class_weights,reduction='mean')

    nodes = [infer(x) for x in phrases]
    nodes = np.array(nodes)  

    graph = Data(x=torch.tensor(nodes,dtype=torch.float).to(device),y=torch.tensor(labels,dtype=torch.long).to(device), edge_index=torch.tensor(edges,dtype=torch.long).to(device))
    graph.n_id = torch.arange(graph.num_nodes)
    mask = torch.tensor([True]*graph.num_nodes, dtype=torch.bool)

    logger.info("Starting GNN training with 22 snapshots")
    for m_n in range(22):

      loader = NeighborLoader(graph, num_neighbors=[-1,-1], batch_size=5000,input_nodes=mask)
      total_loss = 0
      for subg in loader:
          model.train()
          optimizer.zero_grad() 
          out = model(subg.x, subg.edge_index) 
          loss = criterion(out, subg.y) 
          loss.backward() 
          optimizer.step()      
          total_loss += loss.item() * subg.batch_size
      logger.info(f"Snapshot {m_n}/21 - Loss: {total_loss / mask.sum().item():.4f}")

      loader = NeighborLoader(graph, num_neighbors=[-1,-1], batch_size=5000,input_nodes=mask)
      for subg in loader:
          model.eval()
          out = model(subg.x, subg.edge_index)

          sorted, indices = out.sort(dim=1,descending=True)
          conf = (sorted[:,0] - sorted[:,1]) / sorted[:,0]
          conf = (conf - conf.min()) / conf.max()

          pred = indices[:,0]
          cond = (pred == subg.y) | (conf >= 0.9)
          mask[subg.n_id[cond]] = False

      torch.save(model.state_dict(), f'lword2vec_gnn_cadets{m_n}_E3.pth')
      remaining = mask.sum().item()
      if remaining > 0:
        logger.info(f"Model#{m_n} complete: {remaining} nodes still misclassified")
      else:
        logger.info(f"Model#{m_n} complete: All nodes classified!")

In [21]:
from itertools import compress
from torch_geometric import utils

def Get_Adjacent(ids, mapp, edges, hops):
    if hops == 0:
        return set()
    
    neighbors = set()
    for edge in zip(edges[0], edges[1]):
        if any(mapp[node] in ids for node in edge):
            neighbors.update(mapp[node] for node in edge)

    if hops > 1:
        neighbors = neighbors.union(Get_Adjacent(neighbors, mapp, edges, hops - 1))
    
    return neighbors

def calculate_metrics(TP, FP, FN, TN):
    FPR = FP / (FP + TN) if FP + TN > 0 else 0
    TPR = TP / (TP + FN) if TP + FN > 0 else 0

    prec = TP / (TP + FP) if TP + FP > 0 else 0
    rec = TP / (TP + FN) if TP + FN > 0 else 0
    fscore = (2 * prec * rec) / (prec + rec) if prec + rec > 0 else 0

    return prec, rec, fscore, FPR, TPR

def helper(MP, all_pids, GP, edges, mapp):
    logger.info("Calculating evaluation metrics...")
    TP = MP.intersection(GP)
    FP = MP - GP
    FN = GP - MP
    TN = all_pids - (GP | MP)

    two_hop_gp = Get_Adjacent(GP, mapp, edges, 2)
    two_hop_tp = Get_Adjacent(TP, mapp, edges, 2)
    FPL = FP - two_hop_gp
    TPL = TP.union(FN.intersection(two_hop_tp))
    FN = FN - two_hop_tp

    TP, FP, FN, TN = len(TPL), len(FPL), len(FN), len(TN)

    prec, rec, fscore, FPR, TPR = calculate_metrics(TP, FP, FN, TN)
    logger.info(f"True Positives: {TP}, False Positives: {FP}, False Negatives: {FN}")
    console.print(f"[bold green]Precision:[/bold green] {round(prec, 2)}, [bold green]Recall:[/bold green] {round(rec, 2)}, [bold green]Fscore:[/bold green] {round(fscore, 2)}")
    
    return TPL, FPL

In [23]:
f = open("cadets_test.txt")
data = f.read().split('\n')
data = [line.split('\t') for line in data]
df = pd.DataFrame (data, columns = ['actorID', 'actor_type','objectID','object','action','timestamp'])
df = df.dropna()
df.sort_values(by='timestamp', ascending=True,inplace=True)
df = add_attributes(df, "dummy_file")

AttributeError: 'EpochLogger' object has no attribute 'debug'

In [24]:
df = add_attributes(df,"ta1-cadets-e3-official-2.json")

AttributeError: 'EpochLogger' object has no attribute 'debug'

In [25]:
with open("data_files/cadets.json", "r") as json_file:
    GT_mal = set(json.load(json_file))

data = df

phrases,labels,edges,mapp = prepare_graph(data)

nodes = [infer(x) for x in phrases]
nodes = np.array(nodes)  

all_ids = list(data['actorID']) + list(data['objectID'])
all_ids = set(all_ids)

KeyError: 'exec'

In [ ]:
graph = Data(x=torch.tensor(nodes,dtype=torch.float).to(device),y=torch.tensor(labels,dtype=torch.long).to(device), edge_index=torch.tensor(edges,dtype=torch.long).to(device))
graph.n_id = torch.arange(graph.num_nodes)
flag = torch.tensor([True]*graph.num_nodes, dtype=torch.bool)

for m_n in range(22):
  model.load_state_dict(torch.load(f'trained_weights/cadets/lword2vec_gnn_cadets{m_n}_E3.pth'))
  loader = NeighborLoader(graph, num_neighbors=[-1,-1], batch_size=5000)    
  for subg in loader:
      model.eval()
      out = model(subg.x, subg.edge_index)

      sorted, indices = out.sort(dim=1,descending=True)
      conf = (sorted[:,0] - sorted[:,1]) / sorted[:,0]
      conf = (conf - conf.min()) / conf.max()
    
      pred = indices[:,0]
      cond = (pred == subg.y)
      flag[subg.n_id[cond]] = torch.logical_and(flag[subg.n_id[cond]], torch.tensor([False]*len(flag[subg.n_id[cond]]), dtype=torch.bool))
index = utils.mask_to_index(flag).tolist()
ids = set([mapp[x] for x in index])
_ = helper(set(ids),set(all_ids),GT_mal,edges,mapp) 

In [ ]:
def traverse(ids, mapping, edges, hops, visited=None):
    if hops == 0:
        return set()

    if visited is None:
        visited = set()

    neighbors = set()
    for src, dst in zip(edges[0], edges[1]):
        src_mapped, dst_mapped = mapping[src], mapping[dst]

        if (src_mapped in ids and dst_mapped not in visited) or \
           (dst_mapped in ids and src_mapped not in visited):
            neighbors.add(src_mapped)
            neighbors.add(dst_mapped)

        visited.add(src_mapped)
        visited.add(dst_mapped)

    neighbors.difference_update(ids) 
    return ids.union(traverse(neighbors, mapping, edges, hops - 1, visited))

def load_data(file_path):
    with open(file_path, 'r') as file:
        return json.load(file)

def find_connected_alerts(start_alert, mapping, edges, depth, remaining_alerts):
    connected_path = traverse({start_alert}, mapping, edges, depth)
    return connected_path.intersection(remaining_alerts)

def generate_incident_graphs(alerts, edges, mapping, depth):
    incident_graphs = []
    remaining_alerts = set(alerts)

    while remaining_alerts:
        alert = remaining_alerts.pop()
        connected_alerts = find_connected_alerts(alert, mapping, edges, depth, remaining_alerts)

        if len(connected_alerts) > 1:
            incident_graphs.append(connected_alerts)
            remaining_alerts -= connected_alerts

    return incident_graphs